In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.volume import OnBalanceVolumeIndicator, ChaikinMoneyFlowIndicator
from ta.volatility import AverageTrueRange
from ta.volume import VolumeWeightedAveragePrice
from scipy.optimize import minimize
from datetime import datetime
from dateutil.relativedelta import relativedelta
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb



from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
import warnings
warnings.filterwarnings('ignore')



TICKERS = [
        "PETR4.SA", "VALE3.SA", "PRIO3.SA",
        "MGLU3.SA", "LREN3.SA", "ABEV3.SA", "WEGE3.SA",
        "ELET3.SA",
        "SUZB3.SA",
        "EMBR3.SA", "RDOR3.SA", "RAIL3.SA"
    ]


TICKERS_EXPANDIDA = [
    # BANCOS (11)
    "ITUB4.SA",  # Itaú Unibanco
    "BBDC4.SA",  # Bradesco
    "BBAS3.SA",  # Banco do Brasil
    "SANB11.SA", # Santander
    "BPAC11.SA", # Banco do Brasil PN
    "CXSE3.SA",  # Caixa Seguridade
    "BRAP4.SA",  # Bradespar
    "BRSR6.SA",  # Banco do Brasil ON
    "CRFB3.SA",  # Carrefour Brasil
    "PSSA3.SA",  # Porto Seguro
    "PINE4.SA",  # Banco Pine
    
    # ENERGIA (12)
    "PETR4.SA",  # Petrobras PN
    "PRIO3.SA",  # Petrorio
    "OIBR4.SA",  # Oi PN
    "ELET3.SA",  # Eletrobras ON
    "CMIG4.SA",  # Cemig PN
    "CPFE3.SA",  # CPFL Energia
    "EGIE3.SA",  # EDP Energias
    "ENGI11.SA", # Engie Brasil
    "GEMA3.SA",  # Gerdau Metalúrgica
    "LIGHT3.SA", # Light
    "TRPL4.SA",  # Transmissão Paulista
    "EQTL3.SA",  # Equatorial Energia
    
    # MINERAÇÃO (4)
    "VALE3.SA",  # Vale
    "CSNA3.SA",  # Companhia Siderúrgica
    "USIM5.SA",  # Usiminas
    "GGBR4.SA",  # Gerdau PN
    
    # VAREJO (8)
    "MGLU3.SA",  # Magazine Luiza
    "LREN3.SA",  # Lojas Renner
    "ABEV3.SA",  # Ambev
    "RENT3.SA",  # Localiza
    "MOVI3.SA",  # Movida
    "VVAR3.SA",  # Via Varejo
    "PCAR3.SA",  # Impar
    "TRIS3.SA",  # Triscila
    
    # CONSUMO (9)
    "WEGE3.SA",  # WEG
    "JBSS3.SA",  # JBS
    "MSFT34.SA", # Microsoft (ADR)
    "HYPE3.SA",  # Hypera
    "SLCE3.SA",  # SLC Agrícola
    "PETZ3.SA",  # Petz
    "ARZZ3.SA",  # Arezzo
    "TFCO4.SA",  # Telefônico Brasil
    "BRML3.SA",  # Brasil Malha Logística
    
    # TRANSPORTE (6)
    "RAIL3.SA",  # Rumo
    "CCRO3.SA",  # CCR
    "LOGB3.SA",  # Loggi
    "ARZZ3.SA",  # Arezzo (calçados)
    "EMAE4.SA",  # Emae
    "ATUS3.SA",  # Atus
    
    # CONSTRUÇÃO (5)
    "MRVE3.SA",  # MRV Engenharia
    "TEND3.SA",  # Construtora Tenda
    "PLPL3.SA",  # Plano & Plano
    "GFSA3.SA",  # Gafisa
    "TRAD3.SA",  # Tradição
    
    # IMÓVEIS (5)
    "VLID3.SA",  # Validada Imóveis
    "BRIV3.SA",  # BR Imobiliário
    "CYRE3.SA",  # Cyrela
    "EVEN3.SA",  # Even
    "HBOR3.SA",  # Helbor
    
    # COMUNICAÇÃO (3)
    "VIVT3.SA",  # Vivo
    "TIMS3.SA",  # Tim
    "OIBR3.SA",  # Oi ON
    
    # PAPEL E CELULOSE (4)
    "SUZB3.SA",  # Suzano
    "SBSP3.SA",  # Sabesp
    "KLABIN11.SA", # Klabin
    "FIBR3.SA",  # Fibria
    
    # QUÍMICA/HIGIENE (3)
    "TOTS3.SA",  # Totvs
    "BRPR3.SA",  # Brasilfops
    "CLSA3.SA",  # Classa
    
    # ALIMENTOS (4)
    "MBLY3.SA",  # Marfrig
    "BRF3.SA",   # BRF
    "SEQL3.SA",  # Sequoia
    "ASAI3.SA",  # Assaí
    
    # TECNOLOGIA (5)
    "TOTS3.SA",  # Totvs
    "NTCO3.SA",  # Natura
    "BRQT3.SA",  # Brq Digital
    "DIRR3.SA",  # Direcional Engenharia
    "TRPL4.SA",  # Transmissão Paulista
    
    # AVIAÇÃO (3)
    "EMBR3.SA",  # Embraer
    "AZUL4.SA",  # Azul
    "GOLL4.SA",  # Gol
    
    # SEGUROS (3)
    "PSSA3.SA",  # Porto Seguro
    "SULB3.SA",  # Sulamerica
    "SGUP3.SA",  # Seguradoras Unidas
    
    # FINANCEIRAS (4)
    "B3SA3.SA",  # B3
    "MOVI3.SA",  # Movida
    "RBRR3.SA",  # Rede Brasil Real
    "RDOR3.SA",  # Rede D'Or
    
    # AGRONEGÓCIO (3)
    "AGRO3.SA",  # Agrogalaxy
        "AERI3.SA",  # Aerea Invest
    "POSI3.SA",  # Positivo
    ]

In [ ]:
def baixar_e_calcular_indicadores(ticker, start="2010-01-01", end=None):
    """
    Baixa dados e calcula indicadores técnicos.
    """
    try:
        if end is not None:
            data = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        else:
            data = yf.download(ticker, start=start, progress=False, auto_adjust=True)
        
        if data.empty:
            return None
        
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.droplevel(1)
        
        serie = data['Close']
        
        # Indicadores técnicos
        data['SMA_20'] = SMAIndicator(serie, window=20).sma_indicator()
        data['SMA_50'] = SMAIndicator(serie, window=50).sma_indicator()
        data['SMA_200'] = SMAIndicator(serie, window=200).sma_indicator()
        data['EMA_12'] = EMAIndicator(serie, window=12).ema_indicator()
        data['EMA_26'] = EMAIndicator(serie, window=26).ema_indicator()
        data['EMA_50'] = EMAIndicator(serie, window=50).ema_indicator()
        data['RSI_14'] = RSIIndicator(serie, window=14).rsi()
        
        macd = MACD(serie)
        data['MACD'] = macd.macd()
        data['MACD_Hist'] = macd.macd_diff()
        data['ATR_14'] = AverageTrueRange(data['High'], data['Low'], serie, window=14).average_true_range()
        data['ROC_12'] = ROCIndicator(serie, window=12).roc()
        data['OBV'] = OnBalanceVolumeIndicator(serie, data['Volume']).on_balance_volume()
        
        # Retornos e Volatilidade
        data['Ret_1d'] = serie.pct_change(1)
        data['Ret_5d'] = serie.pct_change(5)
        data['Ret_21d'] = serie.pct_change(21)
        data['Vol_21d'] = serie.pct_change().rolling(21).std()
        data['Vol_63d'] = serie.pct_change().rolling(63).std()
        data['Ticker'] = ticker
        
        return data.fillna(method='ffill').dropna()
    except Exception as e:
        print(f"Erro ao baixar {ticker}: {e}")
        return None

def preparar_features_target(df, target_lag=21):
    """
    Prepara features defasadas e target SEM VAZAMENTO DE DADOS.
    
    CORREÇÃO CRÍTICA:
    - Calcula o target ANTES de fazer qualquer shift nas features
    - O target representa o retorno observado 21 dias à frente
    - As features são defasadas em 1 período para não usar info do dia da predição
    - Ao treinar, a data de rebalanceamento não pode ter observado esse target
    """
    data = df.copy()
    
    # 1. PRIMEIRO: Calcular o target futuro (antes de qualquer shift)
    data['Close_Past'] = data['Close'].shift(target_lag)
    data['Target'] = (data['Close'] - data['Close_Past']) / data['Close']
    
    # 2. DEPOIS: Defasar as features em 1 período
    feature_cols = ['SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'EMA_50',
                    'RSI_14', 'MACD', 'MACD_Hist', 'ATR_14', 'ROC_12', 'OBV',
                    'Ret_1d', 'Ret_5d', 'Ret_21d', 'Vol_21d', 'Vol_63d']
    
    for col in feature_cols:
        data[f'{col}_lag1'] = data[col].shift(1)
    
    # 3. REMOVER últimos target_lag linhas (pois não têm target válido)
    data = data.iloc[:-target_lag].copy()
    
    # 4. Usar apenas features defasadas
    feature_cols_lag = [col for col in data.columns if col.endswith('_lag1')]
    data = data.dropna(subset=['Target'] + feature_cols_lag).copy()
    
    return data, feature_cols_lag


def treinar_e_prever(df, feature_cols_lag, data_predicao, ticker, modelo_tipo='RF', target_lag=21):
    """
    Treina modelo SEM VAZAMENTO DE DADOS - CORRIGIDO COM LÓGICA DE PREDIÇÃO.
    
    LÓGICA CRÍTICA:
    - Se queremos PREVER em data_predicao, e temos lag de 21 dias
    - O target de cada linha = retorno 21 dias no FUTURO
    - Portanto, para treinar SEM VAZAMENTO:
      * Treinar APENAS até (data_predicao - target_lag)
      * Isso garante que o target da última linha será observado apenas APÓS data_predicao
    
    - PARA FAZER A PREDIÇÃO:
      * Queremos features do dia data_predicao (ou próximo dia útil)
      * Mas essas features NÃO FORAM CALCULADAS NOS DADOS DE TREINO
      * Precisamos buscar NO DATAFRAME ORIGINAL as features para data_predicao
      * E passar para o modelo treinado
    
    Exemplo:
    - Queremos prever retorno em 31/08/2025
    - Treinar até: 31/08 - 21 = 10/08 (última linha com target conhecido)
    - Predição: usar features de 31/08 (primeira data >= data_predicao)
    - Resultado: previsão para próximos 21 dias (até 21/09)
    """
    # Data limite de treino: (data_predicao - target_lag)
    data_limite_treino = data_predicao - pd.Timedelta(days=target_lag)
    
    # Treinar APENAS com dados ANTES dessa data
    df_treino = df[df.index <= data_limite_treino].copy()
    
    if len(df_treino) < 252:
        print(f" ❌ Dados insuficientes: {len(df_treino)} dias")
        return None
    
    # Extrair X e y para treino
    X_train = df_treino[feature_cols_lag]
    y_train = df_treino['Target'].shift(target_lag)
    
    # Escolher modelo
    if modelo_tipo == 'RF':
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=20,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )
    else:  # OLS
        model = LinearRegression()
    
    model.fit(X_train, y_train)
    
    # ✅ CORREÇÃO: Buscar features para data_predicao no DF ORIGINAL (não no de treino)
    # Encontrar primeira data >= data_predicao
    df_futuro = df[df.index >= data_predicao]
    
    if len(df_futuro) == 0:
        print(f" ❌ Sem dados disponíveis para predição em/após {data_predicao.strftime('%Y-%m-%d')}")
        return None
    
    linha_predicao = df_futuro.iloc[[0]]  # Primeira linha >= data_predicao
    data_real_predicao = linha_predicao.index[0]
    
    # Verificar se features têm NaN
    features_com_nan = linha_predicao[feature_cols_lag].isna().sum().sum()
    if features_com_nan > 0:
        print(f" ⚠ AVISO: {features_com_nan} features com NaN na linha de predição")
    
    # Fazer predição com linha de data_predicao
    X_pred = linha_predicao[feature_cols_lag]
    predicao = model.predict(X_pred)[0]
    
    if predicao > 0:
        print(f"\n{'─'*70}")
        print(f"🔍 MODELO: {modelo_tipo} - {ticker}")
        print(f"📊 DADOS DE TREINO:")
        print(f" • Início: {df_treino.index[0].strftime('%Y-%m-%d')}")
        print(f" • Fim: {df_treino.index[-1].strftime('%Y-%m-%d')} ← ÚLTIMA LINHA COM TARGET CONHECIDO")
        print(f"📊 DADOS DE PREDIÇÃO:")
        print(f" • Data limite treino: {data_limite_treino.strftime('%Y-%m-%d')} (= data_predicao - {target_lag}d)")
        print(f" • Data solicitada: {data_predicao.strftime('%Y-%m-%d')}")
        print(f" • Data real usada: {data_real_predicao.strftime('%Y-%m-%d')} ← FEATURES PARA PREDIÇÃO")
        print(f" • Total dias treino: {len(df_treino)} dias")
        print(f" • Preço em {data_real_predicao.strftime('%Y-%m-%d')}: R$ {linha_predicao['Close'].iloc[0]:.2f}")
        print(f" ✅ Predição: {predicao*100:+.2f}% (para próximos {target_lag} dias)")
    
    return predicao


def calcular_matriz_covariancia(tickers_list, dados_acoes, data_ref, janela=63, target_lag=21):
    """
    Calcula matriz de covariância APENAS com dados históricos.
    
    CORREÇÃO: data_ref deve ser ajustada para (data_ref - target_lag)
    para evitar vazamento temporal
    """
    retornos_hist = []
    
    # Ajustar data de referência: usar dados até (data_ref - target_lag)
    data_limite = data_ref - pd.Timedelta(days=target_lag)
    
    print(f"\nDatas da matriz de covariancia (ajustada para lag={target_lag})")
    print(f"Data referência: {data_ref.strftime('%Y-%m-%d')} → Data limite: {data_limite.strftime('%Y-%m-%d')}")
    
    for ticker in tickers_list:
        df = dados_acoes[ticker]
        # Usar dados até data_limite (não data_ref)
        df_periodo = df[df.index <= data_limite].tail(janela)
        print(f"• {ticker}: {df_periodo.index[0].strftime('%Y-%m-%d')} a {df_periodo.index[-1].strftime('%Y-%m-%d')}")
        
        if len(df_periodo) >= 21:
            retornos = df_periodo['Close'].pct_change().dropna()
            retornos_hist.append(retornos.values)
        else:
            retornos_hist.append(np.zeros(janela-1))
    
    min_len = min(len(r) for r in retornos_hist)
    retornos_df = pd.DataFrame({ticker: r[-min_len:] for ticker, r in zip(tickers_list, retornos_hist)})
    cov_matrix = retornos_df.cov()
    
    return cov_matrix


def otimizar_markowitz(retornos_esperados, cov_matrix, risk_free_rate=0.10/252):
    """
    Otimização de Markowitz: maximizar Sharpe Ratio.
    """
    tickers = list(retornos_esperados.keys())
    n_ativos = len(tickers)
    mu = np.array([retornos_esperados[t] for t in tickers])
    cov = cov_matrix.loc[tickers, tickers].values
    
    def negative_sharpe(weights):
        portfolio_return = np.dot(weights, mu)
        portfolio_vol = np.sqrt(np.dot(weights.T, np.dot(cov, weights)))
        sharpe = (portfolio_return - risk_free_rate) / portfolio_vol if portfolio_vol > 0 else 0
        return -sharpe
    
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = tuple((0, 0.4) for _ in range(n_ativos))
    w0 = np.array([1/n_ativos] * n_ativos)
    
    result = minimize(
        negative_sharpe,
        w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 1000}
    )
    
    if result.success:
        pesos = {ticker: peso for ticker, peso in zip(tickers, result.x) if peso > 0.001}
    else:
        pesos = {ticker: 1/n_ativos for ticker in tickers}
    
    return pesos

def calcular_max_drawdown(retornos):
    """Calcula o máximo drawdown."""
    valor_acumulado = (1 + pd.Series(retornos)).cumprod()
    max_anterior = valor_acumulado.cummax()
    drawdown = (valor_acumulado - max_anterior) / max_anterior
    return drawdown.min()
    

In [6]:
def treinar_e_prever(df, feature_cols_lag, data_predicao, ticker, modelo_tipo='RF', 
                               target_lag=21, usar_validacao=True):
    """
    Treina modelo COM MELHORIAS:
    - Seleção de features (evita ruído)
    - Normalização de features
    - Hiperparâmetros otimizados para RF
    - Validação cruzada temporal
    - Regularização
    """
    
    # Data limite de treino: (data_predicao - target_lag)
    data_limite_treino = data_predicao - pd.Timedelta(days=target_lag)
    
    # Treinar APENAS com dados ANTES dessa data
    df_treino = df[df.index <= data_limite_treino].copy()
    
    if len(df_treino) < 252:
        print(f" ❌ Dados insuficientes: {len(df_treino)} dias")
        return None
    
    # ✅ MELHORIA 1: Seleção de features
    print(f"\n{'─'*70}")
    print(f"🔍 MODELO MELHORADO: {modelo_tipo} - {ticker}")
    print(f"   Selecionando features importantes...")
    
    features_selecionadas, selector = selecionar_features_importantes(
        df_treino, 
        feature_cols_lag, 
        k=min(12, len(feature_cols_lag)//2)  # Usar metade das features
    )
    
    if len(features_selecionadas) < 3:
        print(f" ⚠ Poucas features selecionadas")
        return None
    
    # ✅ MELHORIA 2: Normalização de features
    print(f"   Normalizando features...")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(df_treino[features_selecionadas])
    y_train = df_treino['Target'].values
    
    # ✅ MELHORIA 3: Validação cruzada temporal
    if usar_validacao and len(df_treino) > 500:
        print(f"   Validando com Time Series Split...")
        tscv = TimeSeriesSplit(n_splits=3)
        scores = []
        
        for train_idx, val_idx in tscv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            
            if modelo_tipo == 'RF':
                model_temp = RandomForestRegressor(
                    n_estimators=150,
                    max_depth=8,
                    min_samples_split=30,
                    min_samples_leaf=15,
                    max_features='sqrt',
                    random_state=42,
                    n_jobs=-1
                )
            else:
                model_temp = LinearRegression()
            
            model_temp.fit(X_tr, y_tr)
            score = model_temp.score(X_val, y_val)
            scores.append(score)
        
        cv_score = np.mean(scores)
        print(f"   CV R² Score: {cv_score:.4f}")
    
    # ✅ MELHORIA 4: Hiperparâmetros otimizados para Random Forest
    if modelo_tipo == 'RF':
        model = RandomForestRegressor(
            n_estimators=200,           # ↑ Mais árvores
            max_depth=8,                # ↓ Reduz overfitting
            min_samples_split=30,       # ↑ Mais restritivo
            min_samples_leaf=15,        # ↑ Mais restritivo
            max_features='sqrt',        # Reduz correlação entre árvores
            min_weight_fraction_leaf=0.02,  # Reduz influence de outliers
            random_state=42,
            n_jobs=-1
        )
    else:
        model = LinearRegression()
    
    model.fit(X_train, y_train)
    
    # ✅ MELHORIA 5: Calcular importância de features
    if modelo_tipo == 'RF':
        feature_importance = model.feature_importances_
        top_features_idx = np.argsort(feature_importance)[-5:]
        print(f"   Top 5 features mais importantes:")
        for idx in sorted(top_features_idx, reverse=True):
            print(f"   • {features_selecionadas[idx]}: {feature_importance[idx]:.4f}")
    
    # Buscar features para data_predicao
    df_futuro = df[df.index >= data_predicao]
    
    if len(df_futuro) == 0:
        print(f" ❌ Sem dados disponíveis para predição em/após {data_predicao.strftime('%Y-%m-%d')}")
        return None
    
    linha_predicao = df_futuro.iloc[[0]]
    data_real_predicao = linha_predicao.index[0]
    
    # ✅ Normalizar features de predição com mesma escala
    X_pred = scaler.transform(linha_predicao[features_selecionadas])
    predicao = model.predict(X_pred)[0]
    
    print(f"📊 DADOS DE TREINO:")
    print(f" • Início: {df_treino.index[0].strftime('%Y-%m-%d')}")
    print(f" • Fim: {df_treino.index[-1].strftime('%Y-%m-%d')}")
    print(f" • Total dias: {len(df_treino)} dias")
    print(f"📊 PREDIÇÃO:")
    print(f" • Data solicitada: {data_predicao.strftime('%Y-%m-%d')}")
    print(f" • Data usada: {data_real_predicao.strftime('%Y-%m-%d')}")
    print(f" • Preço: R$ {linha_predicao['Close'].iloc[0]:.2f}")
    print(f" ✅ Predição: {predicao*100:+.2f}% (próximos {target_lag} dias)")
    
    return predicao


def selecionar_features_importantes(df, feature_cols_lag, target_col='Target', k=10):
    """
    Seleciona apenas as k features mais importantes via SelectKBest.
    Reduz overfitting e ruído.
    """
    X = df[feature_cols_lag].fillna(0)
    y = df[target_col]
    
    selector = SelectKBest(score_func=f_regression, k=min(k, len(feature_cols_lag)))
    X_selected = selector.fit_transform(X, y)
    
    # Obter nomes das features selecionadas
    selected_mask = selector.get_support()
    features_selecionadas = [col for col, selected in zip(feature_cols_lag, selected_mask) if selected]
    
    print(f"   Features selecionadas ({len(features_selecionadas)}/{len(feature_cols_lag)}):")
    for feat in features_selecionadas:
        print(f"   • {feat}")
    
    return features_selecionadas, selector


def preparar_features_target(df, target_lag=21):
    """
    Versão melhorada que cria features mais robustas:
    - Adiciona relações entre indicadores
    - Remove features altamente correlacionadas
    - Adiciona volatilidade condicional
    """
    data = df.copy()
    
    # 1. Calcular target
    data['Close_Future'] = data['Close'].shift(-target_lag)
    data['Target'] = (data['Close_Future'] - data['Close']) / data['Close']
    
    # 2. Features base
    feature_cols = ['SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'EMA_50',
                    'RSI_14', 'MACD', 'MACD_Hist', 'ATR_14', 'ROC_12', 'OBV',
                    'Ret_1d', 'Ret_5d', 'Ret_21d', 'Vol_21d', 'Vol_63d']
    
    # 3. ✅ NOVO: Features derivadas (relações)
    # Preço vs média móvel (momentum)
    data['Price_SMA20_Ratio'] = data['Close'] / data['SMA_20'] - 1
    data['Price_SMA50_Ratio'] = data['Close'] / data['SMA_50'] - 1
    
    # Volatilidade relativa
    data['Vol_Ratio_21_63'] = data['Vol_21d'] / (data['Vol_63d'] + 1e-8)
    
    # Força do trend (EMA cruzamentos)
    data['EMA_Crossover_12_26'] = data['EMA_12'] - data['EMA_26']
    
    # RSI momentum
    data['RSI_Velocity'] = data['RSI_14'].diff(5)
    
    feature_cols.extend(['Price_SMA20_Ratio', 'Price_SMA50_Ratio', 'Vol_Ratio_21_63', 
                         'EMA_Crossover_12_26', 'RSI_Velocity'])
    
    # 4. Defasar features
    for col in feature_cols:
        data[f'{col}_lag1'] = data[col].shift(1)
    
    # 5. Remover últimos target_lag
    data = data.iloc[:-target_lag].copy()
    
    # 6. Features defasadas
    feature_cols_lag = [col for col in data.columns if col.endswith('_lag1')]
    data = data.dropna(subset=['Target'] + feature_cols_lag).copy()
    
    return data, feature_cols_lag
    

In [4]:
def estrategia_portfolio_mensal(tickers, data_inicio_teste="2024-01-01",
                                data_fim_teste="2024-12-31", n_acoes=8, 
                                modelo_tipo='RF', salvar_csv=True):
    """
    Estratégia de portfólio mensal com otimização de Markowitz.
    
    Args:
        modelo_tipo: 'RF' para Random Forest ou 'OLS' para Regressão Linear
        salvar_csv: Se True, salva portfólios mensais em CSV
    """
    data_inicio_download = pd.to_datetime(data_inicio_teste) - pd.DateOffset(years=5)
    data_fim_download = pd.to_datetime(data_fim_teste) + pd.DateOffset(days=1)
    
    print(f"\n{'='*80}")
    print(f"ESTRATÉGIA DE PORTFÓLIO MENSAL - MODELO: {modelo_tipo}")
    print(f"{'='*80}")
    print(f"Período de teste: {data_inicio_teste} até {data_fim_teste}")
    print(f"Ações no universo: {n_acoes}")
    print(f"Total de tickers: {len(tickers)}")
    print(f"{'='*80}\n")
    
    print(f"📥 Baixando dados históricos...")
    print(f" • Período: {data_inicio_download.strftime('%Y-%m-%d')} até {data_fim_download.strftime('%Y-%m-%d')}")
    
    dados_acoes = {}
    for ticker in tickers:
        print(f" → {ticker}", end="... ")
        df = baixar_e_calcular_indicadores(
            ticker,
            start=data_inicio_download.strftime('%Y-%m-%d'),
            end=data_fim_download.strftime('%Y-%m-%d')
        )
        
        if df is not None and len(df) > 252:
            ultima_data = df.index[-1]
            if ultima_data > pd.to_datetime(data_fim_teste):
                df = df[df.index <= pd.to_datetime(data_fim_teste)]
            dados_acoes[ticker] = df
            print(f"✓ ({len(df)} dias, até {df.index[-1].strftime('%Y-%m-%d')})")
        else:
            print("✗ (sem dados suficientes)")
    
    print(f"\n✓ Dados baixados para {len(dados_acoes)} ações")
    
    if len(dados_acoes) < n_acoes:
        n_acoes = len(dados_acoes)
    
    # Preparar features
    dados_preparados = {}
    for ticker, df in dados_acoes.items():
        df_prep, feature_cols = preparar_features_target(df, target_lag=21)
        if len(df_prep) > 0:
            dados_preparados[ticker] = (df_prep, feature_cols)
    
    print(f"✓ Features preparadas para {len(dados_preparados)} ações\n")
    
    # Datas de rebalanceamento
    datas_rebalanceamento = pd.date_range(
        start=data_inicio_teste,
        end=data_fim_teste,
        freq='MS'
    )
    
    print(f"📅 Datas de rebalanceamento: {len(datas_rebalanceamento)} meses\n")
    
    resultados_mensais = []
    portfolios_mensais = []  # Para salvar em CSV
    
    # Para cada mês
    for i, data_rebal_original in enumerate(datas_rebalanceamento, 1):
        print(f"\n{'─'*80}")
        print(f"MÊS {i}: {data_rebal_original.strftime('%B/%Y')}")
        print(f"{'─'*80}")
        
        # Ajustar para próximo dia útil
        data_rebal = None
        for offset in range(10):
            data_teste = data_rebal_original + pd.Timedelta(days=offset)
            tem_dados = any(
                len(df[df.index >= data_teste]) > 0
                for df, _ in dados_preparados.values()
            )
            if tem_dados:
                data_rebal = data_teste
                break
        
        if data_rebal is None:
            print(f"⚠ Sem data válida")
            continue
        
        if offset > 0:
            print(f"📅 Data ajustada: {data_rebal.strftime('%Y-%m-%d')} (+{offset} dias)")
        
        # Data fim do mês
        if i < len(datas_rebalanceamento):
            data_fim_mes = datas_rebalanceamento[i]
        else:
            data_fim_mes = pd.to_datetime(data_fim_teste)
        
        # Fazer predições
        predicoes = {}
        for ticker, (df, feature_cols) in dados_preparados.items():
            pred = treinar_e_prever(df, feature_cols, data_rebal, ticker, modelo_tipo=modelo_tipo)
            if pred is not None:
                predicoes[ticker] = pred
        
        print(f"\n✓ Predições realizadas: {len(predicoes)} ações")
        
        if len(predicoes) < 3:
            print(f"⚠ Poucas predições, pulando")
            continue
        
        # Selecionar top N
        predicoes_sorted = sorted(predicoes.items(), key=lambda x: x[1], reverse=True)
        top_acoes = predicoes_sorted[:min(n_acoes, len(predicoes_sorted))]
        
        print(f"\n📊 TOP {len(top_acoes)} AÇÕES:")
        for rank, (ticker, ret_pred) in enumerate(top_acoes, 1):
            print(f" {rank}. {ticker:12s} → Retorno previsto: {ret_pred*100:+.2f}%")
        
        # Covariância
        tickers_top = [t for t, _ in top_acoes]
        cov_matrix = calcular_matriz_covariancia(tickers_top, dados_acoes, data_rebal, janela=63)
        
        # Otimizar
        retornos_esperados = {t: r for t, r in top_acoes}
        pesos_otimos = otimizar_markowitz(retornos_esperados, cov_matrix)
        
        print(f"\n🎯 PESOS OTIMIZADOS:")
        for ticker, peso in sorted(pesos_otimos.items(), key=lambda x: x[1], reverse=True):
            print(f" {ticker:12s} → {peso*100:5.2f}%")
        
        # Salvar portfólio do mês
        portfolio_mes = {
            'Mes': data_rebal_original.strftime('%Y-%m'),
            'Data': data_rebal_original.strftime('%Y-%m-%d'),
            'Modelo': modelo_tipo
        }
        for ticker, peso in pesos_otimos.items():
            portfolio_mes[ticker] = peso
        portfolios_mensais.append(portfolio_mes)
        
        # Calcular retornos reais
        print(f"\n🔍Calculando retornos reais DE {data_rebal.strftime('%Y-%m-%d')} ATÉ {data_fim_mes.strftime('%Y-%m-%d')}")
        
        retornos_reais = {}
        print(f"\n📈 RETORNOS REAIS:")
        
        for ticker in pesos_otimos.keys():
            df_acao = dados_acoes[ticker]
            df_periodo = df_acao[(df_acao.index > data_rebal) & (df_acao.index <= data_fim_mes)]
            
            if len(df_periodo) >= 2:
                preco_inicio = df_periodo['Close'].iloc[0]
                preco_fim = df_periodo['Close'].iloc[-1]
                retorno_real = (preco_fim - preco_inicio) / preco_inicio
                retornos_reais[ticker] = retorno_real
                
                ret_previsto = retornos_esperados[ticker]
                erro = abs(retorno_real - ret_previsto)
                
                print(f" {ticker:12s}: Real={retorno_real*100:+6.2f}% | Previsto={ret_previsto*100:+6.2f}% | Erro={erro*100:5.2f}%")
            else:
                retornos_reais[ticker] = 0
        
        # Retorno do portfólio
        retorno_portfolio = sum(retornos_reais.get(t, 0) * peso for t, peso in pesos_otimos.items())
        
        # Benchmark
        ibov = yf.download("^BVSP", start=data_rebal, end=data_fim_mes, progress=False)
        if not ibov.empty and len(ibov) >= 2:
            if isinstance(ibov['Close'], pd.Series):
                preco_ibov_inicio = ibov['Close'].iloc[0]
                preco_ibov_fim = ibov['Close'].iloc[-1]
            else:
                preco_ibov_inicio = ibov['Close'].iloc[0].values[0] if hasattr(ibov['Close'].iloc[0], 'values') else ibov['Close'].iloc[0]
                preco_ibov_fim = ibov['Close'].iloc[-1].values[0] if hasattr(ibov['Close'].iloc[-1], 'values') else ibov['Close'].iloc[-1]
            retorno_ibov = (preco_ibov_fim - preco_ibov_inicio) / preco_ibov_inicio
        else:
            retorno_ibov = 0
        
        alpha = retorno_portfolio - retorno_ibov
        
        print(f"\n💰 RESULTADO DO MÊS:")
        print(f" Retorno do Portfólio: {retorno_portfolio*100:+.2f}%")
        print(f" Retorno do Ibovespa: {retorno_ibov*100:+.2f}%")
        print(f" Alpha: {alpha*100:+.2f}%")
        
        portfolio_str = ', '.join([f"{t} ({p*100:.1f}%)" for t, p in sorted(pesos_otimos.items(), key=lambda x: x[1], reverse=True)])
        
        resultados_mensais.append({
            'Mes': data_rebal_original.strftime('%Y-%m'),
            'Data': data_rebal_original,
            'Portfolio': portfolio_str,
            'Retorno_Portfolio': retorno_portfolio,
            'Retorno_Ibov': retorno_ibov,
            'Alpha': alpha
        })
    
    # Salvar portfólios em CSV
    if salvar_csv and len(portfolios_mensais) > 0:
        df_portfolios = pd.DataFrame(portfolios_mensais)
        filename = f'portfolios_mensais_{modelo_tipo}_{data_inicio_teste}_{data_fim_teste}.csv'
        df_portfolios.to_csv(filename, index=False)
        print(f"\n✅ Portfólios salvos em: {filename}")
    
    # Resultados finais
    df_resultados = pd.DataFrame(resultados_mensais)
    
    if len(df_resultados) == 0:
        print("\n⚠ Nenhum resultado para processar")
        return None
    
    retorno_acumulado = (1 + df_resultados['Retorno_Portfolio']).prod() - 1
    retorno_ibov_acumulado = (1 + df_resultados['Retorno_Ibov']).prod() - 1
    volatilidade_portfolio = df_resultados['Retorno_Portfolio'].std() * np.sqrt(12)
    sharpe_ratio = (df_resultados['Retorno_Portfolio'].mean() * 12) / volatilidade_portfolio if volatilidade_portfolio > 0 else 0
    max_drawdown = calcular_max_drawdown(df_resultados['Retorno_Portfolio'].values)
    
    print(f"\n\n{'='*80}")
    print(f"RESULTADO FINAL - MODELO: {modelo_tipo}")
    print(f"{'='*80}")
    print(f"Retorno Acumulado do Portfólio: {retorno_acumulado*100:+.2f}%")
    print(f"Retorno Acumulado do Ibovespa: {retorno_ibov_acumulado*100:+.2f}%")
    print(f"Alpha Total: {(retorno_acumulado-retorno_ibov_acumulado)*100:+.2f}%")
    print(f"\nVolatilidade Anualizada: {volatilidade_portfolio*100:.2f}%")
    print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
    print(f"Max Drawdown: {max_drawdown*100:.2f}%")
    print(f"\nMeses Positivos: {(df_resultados['Retorno_Portfolio'] > 0).sum()}/{len(df_resultados)}")
    
    # ============================================================================
    # PREVISÃO FORA DA AMOSTRA - MÊS SEGUINTE AO data_fim_teste
    # ============================================================================
    data_limite = pd.to_datetime(data_fim_teste)
    proximo_mes = data_limite + relativedelta(months=1)
    proximo_mes_inicio = proximo_mes.replace(day=1)
    
    print(f"\n{'='*80}")
    print(f"🔮 PREVISÃO FORA DA AMOSTRA - {proximo_mes_inicio.strftime('%B %Y').upper()}")
    print(f"{'='*80}")
    print(f"⚠ ATENÇÃO: Esta previsão usa APENAS dados até {data_fim_teste}")
    print(f"{'='*80}\n")
    
    print(f"📅 Data limite dos dados: {data_limite.strftime('%Y-%m-%d')}")
    print(f"🎯 Fazendo previsão para: {proximo_mes_inicio.strftime('%B %Y')} (próximos 21 dias úteis)\n")
    
    # Fazer predições para o próximo mês
    predicoes_futuro = {}
    ultimo_dia_treino = {}
    
    print(f"📊 PREDIÇÕES PARA {proximo_mes_inicio.strftime('%B %Y').upper()} (modelo {modelo_tipo}):\n")
    
    for ticker, (df, feature_cols) in dados_preparados.items():
        df_ate_limite = df[df.index <= data_limite]
        
        if len(df_ate_limite) < 252:
            continue
        
        ultimo_dia = df_ate_limite.index[-1]
        ultimo_dia_treino[ticker] = ultimo_dia
        
        # Treinar modelo
        X_train = df_ate_limite[feature_cols]
        y_train = df_ate_limite['Target']
        
        if modelo_tipo == 'RF':
            model = RandomForestRegressor(
                n_estimators=100,
                max_depth=10,
                min_samples_split=20,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1
            )
        else:
            model = LinearRegression()
        
        model.fit(X_train, y_train)
        
        X_pred = df_ate_limite[feature_cols].iloc[[-1]]
        predicao = model.predict(X_pred)[0]
        
        predicoes_futuro[ticker] = predicao
        print(f" {ticker:12s} → Retorno previsto: {predicao*100:+6.2f}% (último treino: {ultimo_dia.strftime('%Y-%m-%d')})")
    
    if len(predicoes_futuro) >= 3:
        # Selecionar top 8
        predicoes_sorted_futuro = sorted(predicoes_futuro.items(), key=lambda x: x[1], reverse=True)
        top_acoes_futuro = predicoes_sorted_futuro[:min(8, len(predicoes_sorted_futuro))]
        
        print(f"\n🏆 TOP 8 AÇÕES RECOMENDADAS PARA {proximo_mes_inicio.strftime('%B %Y').upper()}:")
        for rank, (ticker, ret_pred) in enumerate(top_acoes_futuro, 1):
            ultimo_treino = ultimo_dia_treino[ticker]
            print(f" {rank}. {ticker:12s} → {ret_pred*100:+6.2f}% (dados até {ultimo_treino.strftime('%Y-%m-%d')})")
        
        # Calcular pesos otimizados
        tickers_top_futuro = [t for t, _ in top_acoes_futuro]
        cov_matrix_futuro = calcular_matriz_covariancia(tickers_top_futuro, dados_acoes, data_limite, janela=63)
        retornos_esperados_futuro = {t: r for t, r in top_acoes_futuro}
        pesos_otimos_futuro = otimizar_markowitz(retornos_esperados_futuro, cov_matrix_futuro)
        
        print(f"\n💼 PORTFÓLIO RECOMENDADO (pesos otimizados):")
        retorno_esperado_portfolio = 0
        for ticker, peso in sorted(pesos_otimos_futuro.items(), key=lambda x: x[1], reverse=True):
            ret_esperado = retornos_esperados_futuro[ticker]
            retorno_esperado_portfolio += peso * ret_esperado
            print(f" {ticker:12s} → {peso*100:5.2f}% | Retorno esperado: {ret_esperado*100:+.2f}%")
        
        print(f"\n📈 RETORNO ESPERADO DO PORTFÓLIO: {retorno_esperado_portfolio*100:+.2f}%")
        
        # Salvar previsão futura em CSV
        if salvar_csv:
            portfolio_futuro = {
                'Mes': proximo_mes_inicio.strftime('%Y-%m'),
                'Data': proximo_mes_inicio.strftime('%Y-%m-%d'),
                'Modelo': modelo_tipo,
                'Retorno_Esperado': retorno_esperado_portfolio
            }
            for ticker, peso in pesos_otimos_futuro.items():
                portfolio_futuro[ticker] = peso
            
            filename_futuro = f'previsao_{proximo_mes_inicio.strftime("%Y-%m")}_{modelo_tipo}.csv'
            pd.DataFrame([portfolio_futuro]).to_csv(filename_futuro, index=False)
            print(f"\n✅ Previsão futura salva em: {filename_futuro}")
    else:
        print(f"\n⚠ Poucas predições disponíveis para {proximo_mes_inicio.strftime('%B %Y')}")

In [ ]:
if __name__ == "__main__":
    TICKERS = [
        "PETR4.SA", "VALE3.SA", "PRIO3.SA",
        "MGLU3.SA", "LREN3.SA", "ABEV3.SA", "WEGE3.SA",
        "ELET3.SA",
        "SUZB3.SA",
        "EMBR3.SA", "RDOR3.SA", "RAIL3.SA"
    ]



    TICKERS_EXPANDIDA = [
    # BANCOS (11)
    "ITUB4.SA",  # Itaú Unibanco
    "BBDC4.SA",  # Bradesco
    "BBAS3.SA",  # Banco do Brasil
    "SANB11.SA", # Santander
    "BPAC11.SA", # Banco do Brasil PN
    "CXSE3.SA",  # Caixa Seguridade
    "BRAP4.SA",  # Bradespar
    "BRSR6.SA",  # Banco do Brasil ON
    "CRFB3.SA",  # Carrefour Brasil
    "PSSA3.SA",  # Porto Seguro
    "PINE4.SA",  # Banco Pine
    
    # ENERGIA (12)
    "PETR4.SA",  # Petrobras PN
    "PRIO3.SA",  # Petrorio
    "OIBR4.SA",  # Oi PN
    "ELET3.SA",  # Eletrobras ON
    "CMIG4.SA",  # Cemig PN
    "CPFE3.SA",  # CPFL Energia
    "EGIE3.SA",  # EDP Energias
    "ENGI11.SA", # Engie Brasil
    "GEMA3.SA",  # Gerdau Metalúrgica
    "LIGHT3.SA", # Light
    "TRPL4.SA",  # Transmissão Paulista
    "EQTL3.SA",  # Equatorial Energia
    
    # MINERAÇÃO (4)
    "VALE3.SA",  # Vale
    "CSNA3.SA",  # Companhia Siderúrgica
    "USIM5.SA",  # Usiminas
    "GGBR4.SA",  # Gerdau PN
    
    # VAREJO (8)
    "MGLU3.SA",  # Magazine Luiza
    "LREN3.SA",  # Lojas Renner
    "ABEV3.SA",  # Ambev
    "RENT3.SA",  # Localiza
    "MOVI3.SA",  # Movida
    "VVAR3.SA",  # Via Varejo
    "PCAR3.SA",  # Impar
    "TRIS3.SA",  # Triscila
    
    # CONSUMO (9)
    "WEGE3.SA",  # WEG
    "JBSS3.SA",  # JBS
    "MSFT34.SA", # Microsoft (ADR)
    "HYPE3.SA",  # Hypera
    "SLCE3.SA",  # SLC Agrícola
    "PETZ3.SA",  # Petz
    "ARZZ3.SA",  # Arezzo
    "TFCO4.SA",  # Telefônico Brasil
    "BRML3.SA",  # Brasil Malha Logística
    
    # TRANSPORTE (6)
    "RAIL3.SA",  # Rumo
    "CCRO3.SA",  # CCR
    "LOGB3.SA",  # Loggi
    "ARZZ3.SA",  # Arezzo (calçados)
    "EMAE4.SA",  # Emae
    "ATUS3.SA",  # Atus
    
    # CONSTRUÇÃO (5)
    "MRVE3.SA",  # MRV Engenharia
    "TEND3.SA",  # Construtora Tenda
    "PLPL3.SA",  # Plano & Plano
    "GFSA3.SA",  # Gafisa
    "TRAD3.SA",  # Tradição
    
    # IMÓVEIS (5)
    "VLID3.SA",  # Validada Imóveis
    "BRIV3.SA",  # BR Imobiliário
    "CYRE3.SA",  # Cyrela
    "EVEN3.SA",  # Even
    "HBOR3.SA",  # Helbor
    
    # COMUNICAÇÃO (3)
    "VIVT3.SA",  # Vivo
    "TIMS3.SA",  # Tim
    "OIBR3.SA",  # Oi ON
    
    # PAPEL E CELULOSE (4)
    "SUZB3.SA",  # Suzano
    "SBSP3.SA",  # Sabesp
    "KLABIN11.SA", # Klabin
    "FIBR3.SA",  # Fibria
    
    # QUÍMICA/HIGIENE (3)
    "TOTS3.SA",  # Totvs
    "BRPR3.SA",  # Brasilfops
    "CLSA3.SA",  # Classa
    
    # ALIMENTOS (4)
    "MBLY3.SA",  # Marfrig
    "BRF3.SA",   # BRF
    "SEQL3.SA",  # Sequoia
    "ASAI3.SA",  # Assaí
    
    # TECNOLOGIA (5)
    "TOTS3.SA",  # Totvs
    "NTCO3.SA",  # Natura
    "BRQT3.SA",  # Brq Digital
    "DIRR3.SA",  # Direcional Engenharia
    "TRPL4.SA",  # Transmissão Paulista
    
    # AVIAÇÃO (3)
    "EMBR3.SA",  # Embraer
    "AZUL4.SA",  # Azul
    "GOLL4.SA",  # Gol
    
    # SEGUROS (3)
    "PSSA3.SA",  # Porto Seguro
    "SULB3.SA",  # Sulamerica
    "SGUP3.SA",  # Seguradoras Unidas
    
    # FINANCEIRAS (4)
    "B3SA3.SA",  # B3
    "MOVI3.SA",  # Movida
    "RBRR3.SA",  # Rede Brasil Real
    "RDOR3.SA",  # Rede D'Or
    
    # AGRONEGÓCIO (3)
    "AGRO3.SA",  # Agrogalaxy
    "AERI3.SA",  # Aerea Invest
    "POSI3.SA",  # Positivo
    ]


    # Executar backtest AUDITADO
    resultados = estrategia_portfolio_mensal(
        tickers=TICKERS,
        data_inicio_teste="2025-01-01",
        data_fim_teste="2025-11-23",
        n_acoes=8
    )


ESTRATÉGIA DE PORTFÓLIO MENSAL - MODELO: RF
Período de teste: 2025-01-01 até 2025-11-23
Ações no universo: 8
Total de tickers: 12

📥 Baixando dados históricos...
 • Período: 2020-01-01 até 2025-11-24
 → PETR4.SA... ✓ (1270 dias, até 2025-11-21)
 → VALE3.SA... ✓ (1270 dias, até 2025-11-21)
 → PRIO3.SA... ✓ (1270 dias, até 2025-11-21)
 → MGLU3.SA... ✓ (1270 dias, até 2025-11-21)
 → LREN3.SA... ✓ (1270 dias, até 2025-11-21)
 → ABEV3.SA... ✓ (1270 dias, até 2025-11-21)
 → WEGE3.SA... ✓ (1270 dias, até 2025-11-21)
 → ELET3.SA... ✓ (1270 dias, até 2025-11-21)
 → SUZB3.SA... ✓ (1270 dias, até 2025-11-21)
 → EMBR3.SA... ✓ (1270 dias, até 2025-11-21)
 → RDOR3.SA... ✓ (1032 dias, até 2025-11-21)
 → RAIL3.SA... ✓ (1270 dias, até 2025-11-21)

✓ Dados baixados para 12 ações
✓ Features preparadas para 12 ações

📅 Datas de rebalanceamento: 11 meses


────────────────────────────────────────────────────────────────────────────────
MÊS 1: January/2025
──────────────────────────────────────────────────